In [1]:
import gc
import time
import logging
import os
from datetime import datetime, timedelta

import polars as pl

import sys
sys.path.insert(0, "/home/yusen")

from factor_engine_lazy import FeaturePipeline
from dataloader.jqdata import get_price, get_open_price, get_all_securities, get_call_auction, get_trade_days
from dataloader.api_base.api_conf import auth

In [ ]:
class Config:
    TRADE_DATE = "2026-03-16"
    OUTPUT_DIR = "/home/yusen/ashare_feature/offline_features"
    OUTPUT_PATH = f"{OUTPUT_DIR}/{TRADE_DATE}.parquet"

In [ ]:
# 工具函数
# ============================================================
_trade_days_cache = None

def get_prev_trade_date(date_str: str) -> str:
    """回推前一交易日（基于真实交易日历，正确处理春节等节假日）"""
    global _trade_days_cache
    d = datetime.strptime(date_str, "%Y-%m-%d").date()
    if _trade_days_cache is None:
        trade_days = get_trade_days(
            start_date=(d - timedelta(days=60)).strftime("%Y-%m-%d"),
            end_date=d.strftime("%Y-%m-%d"),
        )
        _trade_days_cache = sorted(trade_days)
        print(f"交易日历已缓存: {_trade_days_cache[0]} ~ {_trade_days_cache[-1]}, 共 {len(_trade_days_cache)} 天")
    prev_days = [td for td in _trade_days_cache if td < d]
    if not prev_days:
        raise ValueError(f"在缓存范围内找不到 {date_str} 之前的交易日")
    return prev_days[-1].strftime("%Y-%m-%d")


def load_auction_data_prev(symbol_arr: list, trade_date: str) -> pl.DataFrame:
    """通过 get_call_auction 读取历史交易日的集合竞价数据。
    current 价格作为 open/close/high/low，time 统一为 trade_date 09:30:00。
    """
    pdf = get_call_auction(start_date=trade_date, end_date=trade_date, symbol=symbol_arr)
    df = pl.from_pandas(pdf)

    # symbol: 取前6位（去掉 .XSHE/.XSHG 后缀）
    df = df.with_columns(
        pl.col("symbol").cast(pl.Utf8).str.slice(0, 6).alias("symbol")
    )

    # 过滤掉 symbol 以 '2' 或 '9' 开头的行
    df = df.filter(
        ~pl.col("symbol").str.starts_with("2") & ~pl.col("symbol").str.starts_with("9")
    )

    # current → open, close, high, low; time → trade_date 09:30:00
    df = df.with_columns([
        pl.lit(datetime.strptime(f"{trade_date} 09:30:00", "%Y-%m-%d %H:%M:%S")).alias("time"),
        pl.col("current").cast(pl.Float64).alias("open"),
        pl.col("current").cast(pl.Float64).alias("close"),
        pl.col("current").cast(pl.Float64).alias("high"),
        pl.col("current").cast(pl.Float64).alias("low"),
        pl.col("volume").cast(pl.Float64),
        pl.col("money").cast(pl.Float64),
    ]).select([
        "time", "symbol", "open", "close", "high", "low", "volume", "money"
    ])

    return df


def load_auction_data_today(symbol_arr: list) -> pl.DataFrame:
    """通过 get_open_price 读取当天集合竞价数据"""
    pdf = get_open_price(symbol_arr)
    df = pl.from_pandas(pdf)

    df = df.with_columns([
        pl.col("time").cast(pl.Utf8).str.strptime(pl.Datetime, "%Y-%m-%d")
            .dt.offset_by("9h30m").alias("time"),
        pl.col("code").cast(pl.Utf8).str.slice(0, 6).alias("symbol"),
    ]).select([
        "time", "symbol", "open", "close", "high", "low", "volume", "money"
    ])

    return df


def load_1min_data(symbols_arr: list, trade_date: str) -> pl.DataFrame:
    """拉取某交易日的 1min 数据"""
    start_dt = datetime.strptime(f"{trade_date} 09:00:00", "%Y-%m-%d %H:%M:%S")
    end_dt = datetime.strptime(f"{trade_date} 15:05:00", "%Y-%m-%d %H:%M:%S")

    df = get_price(
        symbols_arr,
        start_date=start_dt,
        end_date=end_dt,
        frequency='1m',
        count=2000000,
    )

    if not isinstance(df, pl.DataFrame):
        import pandas as pd
        data = {}
        for col in df.columns:
            s = df[col]
            if pd.api.types.is_datetime64_any_dtype(s):
                data[col] = s.dt.to_pydatetime().tolist()
            else:
                data[col] = s.tolist()
        df = pl.DataFrame(data)

        df = df.with_columns([
            pl.col("time").str.to_datetime("%Y-%m-%d %H:%M:%S"),
            pl.col("symbol").cast(pl.Utf8).str.slice(0, 6),
            pl.col("open").cast(pl.Float64),
            pl.col("close").cast(pl.Float64),
            pl.col("high").cast(pl.Float64),
            pl.col("low").cast(pl.Float64),
            pl.col("volume").cast(pl.Float64),
            pl.col("money").cast(pl.Float64),
        ])

    return df


def compute_features(df_all: pl.DataFrame, target_date: str) -> pl.DataFrame:
    """计算特征，只返回目标日期的数据。
    利用 pipeline.run(target_date=...) 在 fillna/drop 之前提前过滤，节省内存。
    """
    pipeline = FeaturePipeline()
    df_input = df_all.rename({"symbol": "code", "time": "datetime"})
    del df_all
    gc.collect()

    df_result = pipeline.run(df_input, target_date=target_date)
    del df_input, pipeline
    gc.collect()

    return df_result

In [ ]:
auth(username='sihang', password='sihang123', env='ali')

trade_date = Config.TRADE_DATE
symbols = get_all_securities(types='stock', date=trade_date)
symbols_arr = list(symbols['symbol'].values)

# 获取前 5 个交易日
prev_dates = []
d = trade_date
for i in range(5):
    d = get_prev_trade_date(d)
    prev_dates.append(d)
prev_dates.reverse()

dfs = []
for prev_date in prev_dates:
    df_prev_auction = load_auction_data_prev(symbols_arr, prev_date)
    df_prev_1m = load_1min_data(symbols_arr, prev_date)
    dfs.extend([df_prev_auction, df_prev_1m])

df_all = pl.concat(dfs)
df_all = df_all.unique(subset=["symbol", "time"], keep="last")
df_all = df_all.sort(["symbol", "time"])

del dfs, df_prev_auction, df_prev_1m
gc.collect()

In [ ]:
is_today = trade_date == datetime.now().strftime("%Y-%m-%d")
if is_today:
    df_auction = load_auction_data_today(symbols_arr)
else:
    df_auction = load_auction_data_prev(symbols_arr, trade_date)

# 补全缺失标的 + current=0 的标的（停牌/未参与竞价），使用昨日收盘价
auction_symbols = set(df_auction["symbol"].to_list())
expected_symbols = set(s[:6] for s in symbols_arr)
missing_symbols = expected_symbols - auction_symbols

# current=0 的标的也需要补全
zero_price_symbols = set(
    df_auction.filter(pl.col("open") == 0.0)["symbol"].to_list()
)
if zero_price_symbols:
    print(f"竞价数据 current=0 的标的: {len(zero_price_symbols)} 个，将用昨日收盘价替换")
    df_auction = df_auction.filter(pl.col("open") != 0.0)

symbols_to_fill = missing_symbols | zero_price_symbols

if symbols_to_fill:
    prev_trade_date_1 = prev_dates[-1]
    prev_close_time = datetime.strptime(f"{prev_trade_date_1} 15:00:00", "%Y-%m-%d %H:%M:%S")
    df_prev_close = df_all.filter(
        (pl.col("time") == prev_close_time) & pl.col("symbol").is_in(list(symbols_to_fill))
    )
    if len(df_prev_close) > 0:
        fill_time = datetime.strptime(f"{trade_date} 09:30:00", "%Y-%m-%d %H:%M:%S")
        df_fill = df_prev_close.select([
            pl.lit(fill_time).alias("time"), pl.col("symbol"),
            pl.col("close").alias("open"), pl.col("close"),
            pl.col("close").alias("high"), pl.col("close").alias("low"),
            pl.lit(0.0).alias("volume"), pl.lit(0.0).alias("money"),
        ])
        df_auction = pl.concat([df_auction, df_fill])
        print(f"已用昨日收盘价补全 {len(df_fill)} 个标的")
    unfilled = symbols_to_fill - set(df_prev_close["symbol"].to_list())
    if unfilled:
        print(f"警告: {len(unfilled)} 个标的在昨日也无收盘价，仍为缺失: {list(unfilled)[:10]}")

df_1m = load_1min_data(symbols_arr, trade_date)
df_all = pl.concat([df_all, df_auction, df_1m])
df_all = df_all.unique(subset=["symbol", "time"], keep="last")
df_all = df_all.sort(["symbol", "time"])

del df_auction, df_1m
if 'df_prev_close' in dir(): del df_prev_close
if 'df_fill' in dir(): del df_fill
gc.collect()

In [ ]:
# 验证 cell 已移除 - df_1m 在上一步已释放
# 如需验证，可在 load_1min_data 之后、del 之前单独检查
print(f"df_all: {df_all.shape}, 内存约 {df_all.estimated_size() / 1024**2:.0f} MB")

In [ ]:
df_features = compute_features(df_all, trade_date)
# compute_features 内部已 del df_all，外部引用已失效
gc.collect()
print(f"df_features: {df_features.shape}, 内存约 {df_features.estimated_size() / 1024**2:.0f} MB")

In [8]:
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
output_path = f"{Config.OUTPUT_DIR}/{trade_date}.parquet"
df_features.write_parquet(output_path)